# GPU Initialization Diagnostics
Use this notebook before large dataset builds to verify CUDA/CuPy health and detect context initialization issues.

This notebook is lightweight and does not load training data arrays.

## Step 1: Environment and Driver Snapshot
Print CUDA-related environment variables and `nvidia-smi` summary.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / 'src').exists():
    project_root = project_root.parent
print('project_root =', project_root)

for key in [
    'CUDA_VISIBLE_DEVICES',
    'NVIDIA_VISIBLE_DEVICES',
    'CUDA_DEVICE_ORDER',
    'CUPY_ACCELERATORS',
    'LD_LIBRARY_PATH',
]:
    print(f"{key}={os.environ.get(key, '<unset>')}")

cmds = [
    ['nvidia-smi', '-L'],
    [
        'nvidia-smi',
        '--query-gpu=index,name,driver_version,persistence_mode,compute_mode,memory.used,memory.total,utilization.gpu',
        '--format=csv,noheader,nounits',
    ],
]
for cmd in cmds:
    print('\n$ ' + ' '.join(cmd))
    out = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(out.stdout.strip() if out.stdout else '<no output>')

project_root = /media/abdul-manan/newvol/Pannuke-project
CUDA_VISIBLE_DEVICES=<unset>
NVIDIA_VISIBLE_DEVICES=<unset>
CUDA_DEVICE_ORDER=<unset>
CUPY_ACCELERATORS=<unset>
LD_LIBRARY_PATH=<unset>

$ nvidia-smi -L
GPU 0: NVIDIA GeForce GTX 1660 Ti (UUID: GPU-1eb075b9-68f1-6ce9-297c-4c46b30b0753)

$ nvidia-smi --query-gpu=index,name,driver_version,persistence_mode,compute_mode,memory.used,memory.total,utilization.gpu --format=csv,noheader,nounits
0, NVIDIA GeForce GTX 1660 Ti, 580.126.09, Disabled, Default, 348, 6144, 11


## Step 2: Run Full Diagnostic Script
Executes `scripts/diagnose_gpu_init.py`, which includes CuPy context init, memory pool reset, and GPU EDT smoke tests.

In [2]:
import subprocess
import sys

cmd = [sys.executable, str(project_root / 'scripts' / 'diagnose_gpu_init.py')]
print('$ ' + ' '.join(cmd))
proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(proc.stdout)
print('Exit code:', proc.returncode)
if proc.returncode != 0:
    print('Diagnostic script reported failures. Check logs above.')

$ /home/abdul-manan/miniconda3/envs/pannuke-ml/bin/python /media/abdul-manan/newvol/Pannuke-project/scripts/diagnose_gpu_init.py
PanNuke GPU initialization diagnostics
Timestamp: 2026-04-22T17:25:35
Python: 3.11.15
Platform: Linux-6.17.0-22-generic-x86_64-with-glibc2.39

Environment variables
CUDA_VISIBLE_DEVICES=<unset>
NVIDIA_VISIBLE_DEVICES=<unset>
CUDA_DEVICE_ORDER=<unset>
CUPY_ACCELERATORS=<unset>
LD_LIBRARY_PATH=<unset>
CONDA_PREFIX=/home/abdul-manan/miniconda3/envs/pannuke-ml

nvidia-smi summary
$ nvidia-smi -L
GPU 0: NVIDIA GeForce GTX 1660 Ti (UUID: GPU-1eb075b9-68f1-6ce9-297c-4c46b30b0753)
$ nvidia-smi --query-gpu=index,name,driver_version,persistence_mode,compute_mode,memory.used,memory.total,utilization.gpu --format=csv,noheader,nounits
0, NVIDIA GeForce GTX 1660 Ti, 580.126.09, Disabled, Default, 354, 6144, 1
$ nvidia-smi --query-compute-apps=gpu_uuid,pid,process_name,used_memory --format=csv,noheader
<no output>

CuPy runtime diagnostics
cupy.__version__ = 14.0.1
CUDA run

## Step 3: In-Session CuPy Pool Cleanup Test
Verifies that CuPy can initialize in this notebook session and that memory pools can be cleared safely.

In [3]:
import traceback

try:
    import cupy as cp
    from cupyx.scipy.ndimage import distance_transform_edt as gpu_edt
    print('cupy version:', cp.__version__)
    print('device count:', cp.cuda.runtime.getDeviceCount())
    _ = cp.cuda.runtime.getDevice()
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()

    mask = cp.zeros((256, 256), dtype=cp.float32)
    mask[96:160, 96:160] = 1.0
    dist = gpu_edt(1.0 - mask)
    print('GPU EDT center distance:', float(dist[128, 128].get()))

    del mask, dist
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()
    print('CuPy pool cleanup OK')
except Exception as exc:
    print('CuPy in-session test failed:', exc)
    traceback.print_exc()

cupy version: 14.0.1
CuPy in-session test failed: cudaErrorUnknown: unknown error


Traceback (most recent call last):
  File "/tmp/ipykernel_30352/1320393576.py", line 7, in <module>
    print('device count:', cp.cuda.runtime.getDeviceCount())
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "cupy_backends/cuda/api/runtime.pyx", line 412, in cupy_backends.cuda.api.runtime.getDeviceCount
  File "cupy_backends/cuda/api/runtime.pyx", line 415, in cupy_backends.cuda.api.runtime.getDeviceCount
  File "cupy_backends/cuda/api/runtime.pyx", line 146, in cupy_backends.cuda.api.runtime.check_status
cupy_backends.cuda.api.runtime.CUDARuntimeError: cudaErrorUnknown: unknown error


## Decision Guide
- If Step 2 and Step 3 pass: run dataset build normally.
- If Step 2 fails with `cudaErrorDevicesUnavailable`: use CPU EDT fallback and continue build.
- If failures persist across fresh sessions: reboot host or reset GPU driver state, then re-run diagnostics.